# AutoGen Agent Chain: Automated Code Testing & Debugging
## A Learning Guide

### What is this?

This notebook demonstrates a **three-agent pipeline** that automates the software QA process:
- One AI agent writes code
- A second AI agent writes tests for that code
- A third AI agent fixes the code if the tests fail

This is a real-world pattern used in automated code review, CI/CD augmentation, and AI-assisted development.

### What you'll learn

- How to create specialist AI agents with distinct roles using AutoGen
- How a `system_message` shapes an agent's behaviour and output format
- How agents hand work off to each other in a sequential pipeline
- How to execute real code programmatically and feed results back into the AI loop
- How to build a self-correcting system that retries on failure

### The pipeline

```
User prompt
    │
    ▼
┌─────────┐    code     ┌────────┐    tests    ┌─────────────┐
│ CodeGen │ ──────────▶ │ Tester │ ──────────▶ │ Test Runner │
└─────────┘             └────────┘             └─────────────┘
                                                      │
                                       pass ◀─────────┤
                                                     fail
                                                      │
                                                      ▼
                                               ┌──────────┐
                                               │ Debugger │ ──▶ (retry loop, max N rounds)
                                               └──────────┘
```

### Prerequisites
- Python 3.10+
- An OpenAI API key in a `.env` file: `OPENAI_API_KEY=sk-...`
- `OPENAI_MODEL_NAME` set to your preferred model (e.g. `gpt-4o-mini`)

---

In [ ]:
%pip install -U "autogen-agentchat" "autogen-ext[openai]"

## Step 1: Install Dependencies

### What these packages provide

| Package | What it gives you |
|---|---|
| `autogen-agentchat` | The `AssistantAgent` class, group chat, and messaging primitives |
| `autogen-ext[openai]` | The `OpenAIChatCompletionClient` that connects agents to the OpenAI API |

You'll also need `python-dotenv` to load your API key from a `.env` file so it never appears in source code. Install it with `pip install python-dotenv` if needed.

---

In [ ]:
import os
import re
import time
import tempfile
import subprocess
import textwrap
import json
import datetime
import pathlib
import sys
import asyncio
from typing import Tuple
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

## Step 2: Imports & Configuration

### Two import groups — and why they're separate

The imports above are **standard library + dotenv** — file I/O, subprocess execution, logging, and environment variables. These are the plumbing; they have no AI dependency.

The imports below are the **AutoGen AI layer**:
- `AssistantAgent` — the base class for all AI-powered agents in this notebook
- `TextMessage` — the message envelope passed between agents and the model client
- `OpenAIChatCompletionClient` — the bridge to the OpenAI Chat Completions API

Keeping them in separate cells makes it easy to swap the AI backend later (e.g. Azure OpenAI or a local Ollama model) without touching the plumbing.

---

In [ ]:
# AutoGen (AG2) import
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
# OpenAI configuration
llm_cfg = {
    "config_list": [
        {
            "model": os.getenv("OPENAI_MODEL_NAME"),
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    ]
}

cfg = llm_cfg["config_list"][0]
model_client = OpenAIChatCompletionClient(
    model=cfg["model"],
    api_key=cfg["api_key"],
)

### Understanding the model client

`OpenAIChatCompletionClient` is created **once** and shared by all three agents. They all talk to the same model through the same connection — their personalities come entirely from their `system_message`, not from using different clients or models.

`llm_cfg` uses the config-list pattern from the older `pyautogen` library. The list structure means you could add a fallback model as a second entry — AutoGen would retry with it if the first one hits a rate limit or error.

> **Checkpoint:** Run the cell above. No output means success — the client is ready. An `AuthenticationError` means your API key is missing or invalid.

---

## Step 3: Infrastructure — Rate Limiting, Logging & Helpers

Before defining the agents, we set up three pieces of supporting infrastructure:

1. **Rate limiting** — a short sleep between API calls prevents hitting OpenAI rate limits during rapid sequential calls. Controlled by the `RATE_SLEEP_SEC` environment variable so it can be tuned without changing code.

2. **Logging** — every agent reply and test result is written to a timestamped `.jsonl` file. In production this feeds into monitoring dashboards. Here it gives you a full audit trail to review after the run.

3. **Helper functions** — three utilities bridge the gap between the AI world (text) and the real world (executing code):

| Function | Purpose |
|---|---|
| `log_event()` | Appends a JSON record to the run log |
| `extract_code_block()` | Pulls the Python code out of an LLM reply that may contain prose and markdown fences |
| `run_py_tests()` | Writes solution + test files to a temp directory and runs them in a subprocess |

`run_py_tests()` is particularly important — it means **real Python actually executes**, not a simulated check. The test results are objective ground truth, not another AI opinion.

In [ ]:
# Light rate limit and retry caps
SLEEP_BETWEEN_CALLS_SEC = float(os.environ.get("RATE_SLEEP_SEC", "1.0"))
MAX_DEBUG_ROUNDS = int(os.environ.get("MAX_DEBUG_ROUNDS", "2"))

# Logging and run artifacts
RUN_DIR = pathlib.Path("runs") / datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_DIR / "log.jsonl"

def log_event(kind: str, payload: dict):
    rec = {"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(), "kind": kind, **payload}
    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Helpers
CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)

def extract_code_block(text: str) -> str:
    m = CODE_BLOCK_RE.search(text or "")
    return textwrap.dedent(m.group(1).strip()) if m else textwrap.dedent(text or "").strip()

def run_py_tests(solution_code: str, test_code: str, timeout_sec: int = 10) -> Tuple[bool, str]:
    with tempfile.TemporaryDirectory() as tmp:
        sol = pathlib.Path(tmp) / "solution.py"
        tst = pathlib.Path(tmp) / "test_solution.py"
        sol.write_text(solution_code, encoding="utf-8")
        tst.write_text(test_code, encoding="utf-8")
        cmd = [sys.executable, str(tst)]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec, cwd=tmp)
            out = (proc.stdout or "") + (proc.stderr or "")
            passed = proc.returncode == 0
            return passed, out.strip()
        except subprocess.TimeoutExpired as e:
            return False, f"TIMEOUT after {timeout_sec}s\n{e}"
        except Exception as e:
            return False, f"EXEC_ERROR: {e}"

async def _ask_async(agent: AssistantAgent, message: str, label: str) -> str:
    await asyncio.sleep(SLEEP_BETWEEN_CALLS_SEC)
    result = await agent.on_messages(
        messages=[TextMessage(content=message, source="user")],
        cancellation_token=None,
    )
    content = ""
    if hasattr(result, "chat_message") and getattr(result.chat_message, "content", None) is not None:
        content = result.chat_message.content
    elif hasattr(result, "messages") and result.messages:
        content = getattr(result.messages[-1], "content", "") or ""
    else:
        content = str(result)
    log_event("llm_reply", {"agent": getattr(agent, "name", "assistant"), "label": label, "content": content})
    return content

def ask(agent: AssistantAgent, message: str, label: str) -> str:
    return asyncio.run(_ask_async(agent, message, label))

## Step 4: Define the Three Specialist Agents

Each agent is an `AssistantAgent` with a carefully crafted `system_message`. The system message is the **job description** — it tells the agent exactly what role it plays, what format its output must follow, and what constraints it must obey.

### Why strict output formatting matters

These agents don't talk to humans — they talk to code. If `CodeGen` adds a sentence like *"Here is your code:"* before the code block, `extract_code_block()` will still find and extract it. But consistent formatting means fewer edge cases and more reliable pipelines.

### The three roles at a glance

| Agent | Receives | Produces | Key constraint |
|---|---|---|---|
| `CodeGen` | Task description | `solution.py` as a code block | One function only, no comments, no extra text |
| `Tester` | Task description | `test_solution.py` as a code block | Uses `assert` statements, covers edge cases |
| `Debugger` | Failing test output + current code | Fixed `solution.py` as a code block | Must keep the same function signature |

### Notice: all three share the same `model_client`

The agents are specialised by their `system_message`, not by using different models. This is a key insight — **role separation comes from instructions, not from different AI systems**.

> **Exercise:** Read each `system_message` carefully. What would happen if you removed the instruction *"Reply with ONLY a Python fenced code block"*? How would that affect the rest of the pipeline?

In [ ]:
codegen = AssistantAgent(
    name="CodeGen",
    system_message=(
        "You are CodeGen. Write Python code for solution.py with a SINGLE function:\n"
        "def is_palindrome(text: str) -> bool\n"
        "- Return True if text is a palindrome.\n"
        "- Ignore non-alphanumeric characters and case.\n"
        "- Do not print; no comments; no extra text.\n"
        "Reply with ONLY a Python fenced code block for solution.py."
    ),
    model_client=model_client,
)

tester = AssistantAgent(
    name="Tester",
    system_message=(
        "You are Tester. Write tests in test_solution.py using Python asserts.\n"
        "Import from solution import is_palindrome.\n"
        "Cover typical and edge cases (empty, punctuation, mixed case, long string).\n"
        "Reply with ONLY a Python fenced code block for test_solution.py."
    ),
    model_client=model_client,
)

debugger = AssistantAgent(
    name="Debugger",
    system_message=(
        "You are Debugger. Given failing test output and the current solution.py, "
        "produce a FIXED solution.py that passes tests. Keep the same signature:\n"
        "def is_palindrome(text: str) -> bool\n"
        "Reply with ONLY a Python fenced code block for solution.py."
    ),
    model_client=model_client,
)

## Step 5: The Orchestrator — `main()`

The orchestrator is the conductor that calls each agent in sequence, runs the real tests, and decides whether to loop back through the Debugger.

### The five stages

```
Stage 1 ─── CodeGen writes solution.py
Stage 2 ─── Tester writes test_solution.py
Stage 3 ─── Test Runner executes tests (real Python subprocess — no AI involved here)
Stage 4 ─── If tests fail: Debugger fixes solution.py → loop back to Stage 3
Stage 5 ─── Save artifacts, log outcome, report result
```

### Key design decisions to notice as you read the code

- **Real code execution** happens in a temporary directory via `subprocess` — the AI has no access to the file system and cannot cheat
- **The retry loop** is bounded by `MAX_DEBUG_ROUNDS` to prevent infinite loops if the Debugger cannot fix the problem
- **Context is explicit** — when the Debugger is called, it receives both the failing test output *and* the current source, giving it everything it needs to produce a fix
- **The retry guard** (`if fixed and fixed != solution`) prevents wasted API calls if the Debugger returns the same code unchanged
- **Every event is logged** — if the run fails, the `.jsonl` log has a full audit trail

> **Before running:** Confirm your `.env` file has both `OPENAI_API_KEY` and `OPENAI_MODEL_NAME` set.

In [ ]:
async def main():
    print("== AutoGen Code Testing and Debugging Chain ==")
    print(f"Deployment: {os.getenv('OPENAI_MODEL_NAME')}")
    log_event("start", {"deployment": os.getenv("OPENAI_MODEL_NAME")})

    # 1) CodeGen creates solution.py
    cg_out = await _ask_async(codegen, "Write solution.py as specified. Return ONLY the code block.", "codegen")
    solution = extract_code_block(cg_out)
    if "def is_palindrome" not in solution:
        cg_out = await _ask_async(codegen, "Please include def is_palindrome(text: str) -> bool", "codegen_retry")
        solution = extract_code_block(cg_out)
    log_event("solution", {"code": solution})
    print("\n[CodeGen] Generated solution.py")

    # 2) Tester creates test_solution.py
    tt_out = await _ask_async(tester, "Create test_solution.py for is_palindrome; ONLY code block.", "tester")
    tests = extract_code_block(tt_out)
    log_event("tests", {"code": tests})
    print("[Tester] Generated test_solution.py")

    # 3) Execute tests
    passed, output = run_py_tests(solution, tests, timeout_sec=10)
    print("\n[Test Runner] First run passed? ", passed)
    print(output)
    log_event("test_run", {"passed": passed, "output": output})

    # 4) If failed, loop with Debugger
    rounds = 0
    while not passed and rounds < MAX_DEBUG_ROUNDS:
        rounds += 1
        print(f"\n[Debugger] Fix attempt #{rounds}")
        dbg_prompt = (
            "Tests failed with output below. Current solution.py is also provided.\n\n"
            f"=== FAIL LOG ===\n{output}\n\n=== CURRENT solution.py ===\n```python\n{solution}\n```"
        )
        dbg_out = await _ask_async(debugger, dbg_prompt, f"debugger_round_{rounds}")
        fixed = extract_code_block(dbg_out)
        if fixed and fixed != solution:
            solution = fixed
            log_event("debug_fix", {"round": rounds, "code": solution})
        passed, output = run_py_tests(solution, tests, timeout_sec=10)
        print("[Test Runner] Passed after fix? ", passed)
        print(output)
        log_event("test_rerun", {"round": rounds, "passed": passed, "output": output})

    # 5) Save artifacts and finish
    (RUN_DIR / "solution.py").write_text(solution, encoding="utf-8")
    (RUN_DIR / "test_solution.py").write_text(tests, encoding="utf-8")
    status = "ALL TESTS PASSED" if passed else "STOPPED WITH FAILURES (see logs)"
    print("\n== RESULT:", status)
    print("Artifacts saved to:", RUN_DIR)
    log_event("end", {"status": status, "artifacts_dir": str(RUN_DIR)})

await main()

---

## What just happened? — Walkthrough

After running `main()`, scroll back through the output and trace each stage:

1. **`[CodeGen]`** — the LLM wrote an `is_palindrome` implementation. It had no idea what tests would be written against it.
2. **`[Tester]`** — a *different* agent (same model, different system message) wrote tests without seeing the implementation.
3. **`[Test Runner]`** — pure Python ran those tests in a subprocess. This is the only stage with **no AI involvement** — the result is objective.
4. **`[Debugger]`** *(if it appeared)* — received the failure output and the source, and produced a corrected version. The loop ran again until tests passed or the round limit was hit.

The key insight: **each agent only sees what it needs to**. CodeGen never sees the tests. Tester never sees the implementation. The Debugger sees both, but only because it needs both to fix the problem. This separation of concerns is what makes the pipeline more reliable than a single "do everything" prompt.

---

## Inspect the artifacts

A timestamped `runs/` directory was created when you ran `main()`. Open it to find:
- `solution.py` — the final passing implementation
- `test_solution.py` — the AI-generated test file
- `log.jsonl` — a line-by-line audit trail of every LLM call and test run (open with any text editor)

---

## Key Takeaways

| Concept | What it means in this notebook |
|---|---|
| **`AssistantAgent`** | An AI with a fixed role defined by its `system_message` |
| **`system_message`** | The job description — shapes personality, output format, and constraints |
| **Shared `model_client`** | One connection, multiple personalities — role comes from instructions not models |
| **`_ask_async`** | The wrapper that sends a message to an agent and waits for the reply |
| **Subprocess execution** | Tests run in real Python — objective pass/fail, no AI guessing |
| **Bounded retry loop** | `MAX_DEBUG_ROUNDS` prevents infinite loops; the system fails gracefully |
| **Audit log** | Every event written to `.jsonl` — production systems need traceability |

---

## Exercises

### Beginner
1. Change `MAX_DEBUG_ROUNDS` to `0` and run again. What happens when tests fail with no Debugger rounds allowed?
2. Change the task from `is_palindrome` to `is_prime`. Update all three `system_message` strings accordingly.

### Intermediate
3. Add a fourth agent — a `Reviewer` — that reads the final passing `solution.py` and comments on code quality. Call it after Stage 5.
4. Modify `log_event` to also print a one-line summary to the console each time it's called, so you can watch the audit trail grow in real time.

### Advanced
5. Replace the `assert`-based test style with `pytest`. Modify `run_py_tests()` to invoke `pytest` instead of running the file directly, and update the `Tester` system message accordingly.
6. Make the pipeline work on **any** function spec provided at runtime via `input()`, rather than the hardcoded `is_palindrome` task.